# Notebook 09 – Multimodal GRU Fusion (Phase 3–5)

This notebook runs the full **5-config ablation study** for the intermediate-fusion multimodal GRU and compares against the Tier-2 unimodal GRU baseline (ROC-AUC 0.717, PR-AUC 0.773).

## Architecture
```
Smartwatch sequence  (batch, 288, 4)   ← subsampled 5× from 1440 min
    GRU(hidden=64, layers=2, dropout=0.3)
    h_T                                (batch, 64)

Tabular features     (batch, K)        ← K = 0 … 90 depending on config
    MLP(K→64→32, ReLU, dropout=0.3)   ← omitted when K=0
    tab_embed                          (batch, 32)

Fusion:  concat([h_T, tab_embed])      (batch, 96)  [or 64 for SW-only]
    Linear(96→1) → logit
```

## Ablation configs
| Config | Tab sources | K |
|---|---|---|
| `SW_only` | — | 0 |
| `SW_inhaler` | Smartinhaler | 6 |
| `SW_inhaler_pef` | + Peak flow | 21 |
| `SW_inhaler_pef_dailyQ` | + Daily questionnaire | 24 |
| `Full` | + Weekly Q + Patient static | 90 |

## Key design notes (see `notes.txt`)
- **pos_weight = 3.43** (train positive rate 22.5%), identical across all configs (same labels)
- **Platt scaling** on validation probabilities corrects the train→test prevalence shift (22.5% → 48.2%)
- **Primary metric = ROC-AUC**; PR-AUC and Brier are inflated by the high test-set prevalence
- Missing rates: smartinhaler 63.1%, peakflow 42.6%, daily Q 84.4% — sentinel (-1) + missing indicator already baked in

## 0. Environment Setup

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
COLAB_REPO_DIR = Path("/content/EECE-693-project")

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate
    if COLAB_REPO_DIR.exists() and (COLAB_REPO_DIR / "src" / "config.py").exists():
        return COLAB_REPO_DIR
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "Could not find the project root. Run from the repo root or notebooks/ folder."
        ) from exc
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    return COLAB_REPO_DIR

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "PyTorch is not installed. Install with: pip install torch --index-url https://download.pytorch.org/whl/cpu"
    ) from exc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score

from src.config import OUTPUT_TABLES, OUTPUT_FIGURES
from src.deep_learning import (
    MULTIMODAL_ABLATION_CONFIGS,
    run_multimodal_experiment,
    prepare_multimodal_data,
    make_multimodal_gru,
    make_multimodal_loaders,
    train_multimodal_model,
    predict_multimodal_model,
    _select_tab_cols,
    _platt_scale,
)

print(f"Torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
print(f"Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Run Full Ablation Study

Single call — loads data once, trains 5 models, Platt-scales each, saves results.

In [ ]:
EPOCHS   = 20
BATCH    = 64
PATIENCE = 5

results = run_multimodal_experiment(
    epochs=EPOCHS,
    batch_size=BATCH,
    patience=PATIENCE,
    random_state=42,
    output_path=OUTPUT_TABLES / "multimodal_results.csv",
)
results

## 2. Compare Against Unimodal GRU Baseline

In [ ]:
BASELINE_GRU = {
    "model": "GRU (2-layer) – Tier-2 baseline",
    "roc_auc": 0.717,
    "pr_auc":  0.773,
    "f1":      0.645,
    "recall":  0.619,
    "precision": 0.673,
    "brier":   0.220,
}

display_cols = ["model", "roc_auc", "pr_auc", "f1", "precision", "recall", "brier", "tab_dim", "best_epoch"]

# Append baseline row for comparison
baseline_row = pd.DataFrame([BASELINE_GRU])
compare = pd.concat([results[display_cols], baseline_row[display_cols].reindex(columns=display_cols)], ignore_index=True)

# Highlight best ROC-AUC
best_auc = compare["roc_auc"].max()
compare["Δ ROC-AUC vs baseline"] = (compare["roc_auc"] - BASELINE_GRU["roc_auc"]).round(4)
compare["Δ PR-AUC vs baseline"]  = (compare["pr_auc"]  - BASELINE_GRU["pr_auc"]).round(4)

print("Results (primary metric = ROC-AUC; PR-AUC inflated by 48% test prevalence)")
print("=" * 80)
compare

## 3. ROC-AUC Ablation Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

colors = ["#4C72B0"] * len(results) + ["#C44E52"]
labels = list(results["model"]) + ["GRU baseline (Tier-2)"]
aucs   = list(results["roc_auc"]) + [BASELINE_GRU["roc_auc"]]

bars = ax.barh(labels[::-1], aucs[::-1], color=colors[::-1], edgecolor="white", height=0.55)
ax.axvline(BASELINE_GRU["roc_auc"], color="#C44E52", linestyle="--", lw=1.5, label="Tier-2 GRU baseline")

for bar, val in zip(bars, aucs[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

ax.set_xlim(0.4, max(aucs) + 0.08)
ax.set_xlabel("ROC-AUC")
ax.set_title("Multimodal Ablation – ROC-AUC (test set, Platt-calibrated)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "multimodal_ablation_roc_auc.png", dpi=150)
plt.show()
print("Saved: outputs/figures/multimodal_ablation_roc_auc.png")

## 4. PR-AUC Bar Chart

> **Note:** PR-AUC and Brier are inflated because the test set has a 48.2% positive rate (vs 22.5% in training). No-skill PR-AUC = 0.482. Use ROC-AUC as the primary comparison metric.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

pr_aucs = list(results["pr_auc"]) + [BASELINE_GRU["pr_auc"]]
bars = ax.barh(labels[::-1], pr_aucs[::-1], color=colors[::-1], edgecolor="white", height=0.55)

# No-skill line = test prevalence
TEST_PREVALENCE = 0.482
ax.axvline(TEST_PREVALENCE, color="grey", linestyle=":", lw=1.5, label=f"No-skill (prevalence={TEST_PREVALENCE})")
ax.axvline(BASELINE_GRU["pr_auc"], color="#C44E52", linestyle="--", lw=1.5, label="Tier-2 GRU baseline")

for bar, val in zip(bars, pr_aucs[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

ax.set_xlim(0.3, max(pr_aucs) + 0.1)
ax.set_xlabel("PR-AUC")
ax.set_title("Multimodal Ablation – PR-AUC (test set, Platt-calibrated)\n"
             "⚠ Inflated by 48.2% test prevalence — compare relative differences only")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "multimodal_ablation_pr_auc.png", dpi=150)
plt.show()
print("Saved: outputs/figures/multimodal_ablation_pr_auc.png")

## 5. Training Curves for All Configs

In [ ]:
fig, axes = plt.subplots(1, len(MULTIMODAL_ABLATION_CONFIGS), figsize=(16, 3.5), sharey=False)

for ax, (config_name, _) in zip(axes, MULTIMODAL_ABLATION_CONFIGS):
    hist_path = OUTPUT_TABLES / f"mm_{config_name.lower()}_history.csv"
    if not hist_path.exists():
        ax.text(0.5, 0.5, "No history\n(not yet run)", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(config_name, fontsize=9)
        continue
    hist = pd.read_csv(hist_path)
    ax.plot(hist["epoch"], hist["train_loss"], label="train", lw=1.5)
    ax.plot(hist["epoch"], hist["val_loss"],   label="val",   lw=1.5, linestyle="--")
    best_ep = results.loc[results["model"] == config_name, "best_epoch"]
    if not best_ep.empty:
        ax.axvline(int(best_ep.iloc[0]), color="grey", linestyle=":", lw=1, label=f"best epoch {int(best_ep.iloc[0])}")
    ax.set_title(config_name, fontsize=9)
    ax.set_xlabel("Epoch", fontsize=8)
    ax.set_ylabel("BCE Loss", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

plt.suptitle("Training Curves – Multimodal Ablation Configs", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "multimodal_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/figures/multimodal_training_curves.png")

## 6. ROC Curves – All Configs on Test Set

Re-trains each config from scratch to get test-set predictions, then overlays all ROC curves.

In [ ]:
print("Loading multimodal data for ROC curve plotting (uses cached data from Step 1)…")
split = prepare_multimodal_data(random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
palette = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

for (config_name, prefixes), color in zip(MULTIMODAL_ABLATION_CONFIGS, palette):
    col_indices, _ = _select_tab_cols(split.tab_feature_names, prefixes)
    tab_dim = len(col_indices)

    model = make_multimodal_gru(tab_dim=tab_dim)
    tr_l, va_l, te_l = make_multimodal_loaders(split, col_indices=col_indices, batch_size=64)
    model, _, _ = train_multimodal_model(
        model, tr_l, va_l, split.y_train,
        epochs=EPOCHS, patience=PATIENCE,
    )
    p_val  = predict_multimodal_model(model, va_l)
    p_test = predict_multimodal_model(model, te_l)
    p_cal  = _platt_scale(split.y_val, p_val, p_test)

    fpr, tpr, _ = roc_curve(split.y_test, p_cal)
    auc = roc_auc_score(split.y_test, p_cal)
    ax.plot(fpr, tpr, label=f"{config_name} (AUC={auc:.3f})", color=color, lw=1.8)

ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="No skill")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves – Multimodal Ablation (test set, Platt-calibrated)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "multimodal_roc_curves.png", dpi=150)
plt.show()
print("Saved: outputs/figures/multimodal_roc_curves.png")

## 7. Threshold Analysis – Best Config

Identifies the F1-optimal threshold and two clinical operating points (high-recall, high-precision) for the best-performing ablation config.

In [ ]:
best_config_name = results.loc[results["roc_auc"].idxmax(), "model"]
best_prefixes    = dict(MULTIMODAL_ABLATION_CONFIGS)[best_config_name]
print(f"Best config by ROC-AUC: {best_config_name!r}")

col_indices, _ = _select_tab_cols(split.tab_feature_names, best_prefixes)
tab_dim = len(col_indices)

model = make_multimodal_gru(tab_dim=tab_dim)
tr_l, va_l, te_l = make_multimodal_loaders(split, col_indices=col_indices, batch_size=64)
model, _, _ = train_multimodal_model(
    model, tr_l, va_l, split.y_train,
    epochs=EPOCHS, patience=PATIENCE,
)
p_val  = predict_multimodal_model(model, va_l)
p_test = predict_multimodal_model(model, te_l)
p_cal  = _platt_scale(split.y_val, p_val, p_test)

from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.05, 0.95, 0.01)
f1s, precs, recs = [], [], []
for t in thresholds:
    pred = (p_cal >= t).astype(int)
    f1s.append(f1_score(split.y_test, pred, zero_division=0))
    precs.append(precision_score(split.y_test, pred, zero_division=0))
    recs.append(recall_score(split.y_test, pred, zero_division=0))

best_t_idx = int(np.argmax(f1s))
best_t     = thresholds[best_t_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1s,   label="F1",        lw=2)
ax.plot(thresholds, precs, label="Precision",  lw=1.5, linestyle="--")
ax.plot(thresholds, recs,  label="Recall",     lw=1.5, linestyle=":")
ax.axvline(best_t, color="grey", linestyle="-.", lw=1.2,
           label=f"F1-optimal threshold = {best_t:.2f}")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis – {best_config_name} (test set)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "multimodal_threshold_analysis.png", dpi=150)
plt.show()

print(f"\nF1-optimal threshold : {best_t:.2f}")
print(f"  Precision={precs[best_t_idx]:.3f}  Recall={recs[best_t_idx]:.3f}  F1={f1s[best_t_idx]:.3f}")
print()

# Clinical operating points
rec_idx  = next((i for i, r in enumerate(recs) if r >= 0.80), best_t_idx)
prec_idx = next((i for i, p in enumerate(precs) if p >= 0.70), best_t_idx)
for label, idx in [("High recall (≥80%)", rec_idx), ("High precision (≥70%)", prec_idx)]:
    print(f"{label}: threshold={thresholds[idx]:.2f}  "
          f"P={precs[idx]:.3f}  R={recs[idx]:.3f}  F1={f1s[idx]:.3f}")

## 8. Missing-Modality Impact Analysis

Inspects what fraction of test windows have each modality available, and correlates modality presence with model confidence.

In [ ]:
missing_cols = [n for n in split.tab_feature_names if n.endswith("_missing")]
test_tab = split.tab_test   # already z-scored; missing indicator cols are still binary after z-score

# Recover raw binary indicators (original before z-score): abs(val) near 1 → missing, near 0 → present
# Better: use the original feature names to identify missing indicator column indices
missing_indices = [split.tab_feature_names.index(c) for c in missing_cols]

print("Missing modality rates in TEST set (from standardised indicators):")
for col, idx in zip(missing_cols, missing_indices):
    # In the standardised space the indicator is no longer exactly 0/1, so
    # we threshold at the midpoint between z(0) and z(1).
    vals = test_tab[:, idx]
    rate = (vals > vals.mean()).mean()
    print(f"  {col:35s}: {rate*100:.1f}% missing")

## 9. Save Full Results Table

In [ ]:
# Load Tier-2 all_model_results for side-by-side comparison
all_path = OUTPUT_TABLES / "all_model_results.csv"
if all_path.exists():
    tier2 = pd.read_csv(all_path)
    combined = pd.concat([tier2, results[["model", "roc_auc", "pr_auc", "f1", "precision", "recall", "accuracy", "brier"]]], ignore_index=True)
    combined.to_csv(OUTPUT_TABLES / "all_model_results_with_multimodal.csv", index=False)
    print(f"Saved combined results → {OUTPUT_TABLES / 'all_model_results_with_multimodal.csv'}")
    display(combined)
else:
    print("all_model_results.csv not found — saving multimodal results only")
    results.to_csv(OUTPUT_TABLES / "multimodal_results.csv", index=False)

print("\nMultimodal results CSV:")
display(results)

## 10. Summary

Prints a concise human-readable summary for copy-paste into the report.

In [ ]:
print("=" * 70)
print("MULTIMODAL ABLATION SUMMARY")
print("=" * 70)
print(f"{'Config':<30} {'K':>4}  {'ROC-AUC':>8}  {'PR-AUC':>7}  {'F1':>6}  {'Epoch':>5}")
print("-" * 70)

for _, row in results.iterrows():
    print(f"{row['model']:<30} {int(row['tab_dim']):>4}  "
          f"{row['roc_auc']:>8.3f}  {row['pr_auc']:>7.3f}  {row['f1']:>6.3f}  "
          f"{int(row['best_epoch']):>5}")

print("-" * 70)
print(f"{'GRU baseline (Tier-2)':<30} {'–':>4}  "
      f"{BASELINE_GRU['roc_auc']:>8.3f}  {BASELINE_GRU['pr_auc']:>7.3f}  "
      f"{BASELINE_GRU['f1']:>6.3f}  {'–':>5}")
print("=" * 70)

best_row = results.loc[results["roc_auc"].idxmax()]
delta = best_row["roc_auc"] - BASELINE_GRU["roc_auc"]
print(f"\nBest multimodal config : {best_row['model']}")
print(f"  ROC-AUC  = {best_row['roc_auc']:.3f}  (Δ vs Tier-2 GRU = {delta:+.3f})")
print(f"  PR-AUC   = {best_row['pr_auc']:.3f}  (⚠ inflated by 48% test prevalence)")
print(f"  F1       = {best_row['f1']:.3f}")
print(f"  tab_dim  = {int(best_row['tab_dim'])}")
print()
print("Note: Platt scaling applied on validation set before all test-set evaluations.")
print("Note: ROC-AUC is the primary metric; PR-AUC and Brier inflated by high test prevalence.")